In [ ]:
import pandas as pd
import random

csv_path = "VERI_metadata.csv"  # change if needed
output_path = "veri_image_captions_with_switch.csv"

random_seed = 128
random.seed(random_seed)

df = pd.read_csv(csv_path)
df["caption"] = df["caption"].fillna("").astype(str)

df["condition"] = df["image_id"].str.extract(r"_(safe|danger)$")
df["base_id"] = df["image_id"].str.replace(r"_(safe|danger)$", "", regex=True)

# Paired safe/danger caption
caption_lookup = {
    (row["base_id"], row["condition"]): row["caption"]
    for _, row in df.iterrows()
}

def opposite_condition(condition):
    if condition == "safe":
        return "danger"
    if condition == "danger":
        return "safe"
    return None

def get_switch_caption(row):
    opposite = opposite_condition(row["condition"])
    if opposite is None:
        return ""
    return caption_lookup.get((row["base_id"], opposite), "")

df["switch_caption"] = df.apply(get_switch_caption, axis=1)

# Helper: assign random captions without replacement under a constraint
def assign_random_column(df, column_name, is_valid_candidate):
    df[column_name] = ""

    available_indices = list(df.index)
    random.shuffle(available_indices)

    for idx in df.index:
        valid_choices = [
            candidate_idx
            for candidate_idx in available_indices
            if is_valid_candidate(idx, candidate_idx)
        ]

        if not valid_choices:
            raise ValueError(
                f"No valid caption available for row {idx} "
                f"when creating {column_name}."
            )

        chosen_idx = valid_choices[0]
        df.at[idx, column_name] = df.at[chosen_idx, "caption"]
        available_indices.remove(chosen_idx)

# 1. Totally random caption from another pair
assign_random_column(
    df,
    "random_caption",
    lambda idx, cand: df.at[cand, "base_id"] != df.at[idx, "base_id"]
)

# 2. Same risk: safe -> other safe, danger -> other danger
assign_random_column(
    df,
    "random_same_risk",
    lambda idx, cand: (
        df.at[cand, "base_id"] != df.at[idx, "base_id"]
        and df.at[cand, "condition"] == df.at[idx, "condition"]
    )
)

# 3. Opposite risk: safe -> other danger, danger -> other safe
assign_random_column(
    df,
    "random_opposite_risk",
    lambda idx, cand: (
        df.at[cand, "base_id"] != df.at[idx, "base_id"]
        and df.at[cand, "condition"] == opposite_condition(df.at[idx, "condition"])
    )
)

df = df.drop(columns=["condition", "base_id"])

check_cols = [
    "caption",
    "switch_caption",
    "random_caption",
    "random_same_risk",
    "random_opposite_risk"
]

def check_caption_uniqueness(row):
    values = [
        str(row[col]).strip()
        for col in check_cols
        if pd.notna(row[col]) and str(row[col]).strip()
    ]

    if len(values) != len(check_cols):
        print(f"Row {row['image_id']} is missing captions: {values}")
        return "FAIL_missing_caption"

    if len(values) != len(set(values)):
        print(f"Row {row['image_id']} has duplicate captions: {values}")
        return "FAIL_duplicate_caption"

    return "PASS"

df["caption_swap_check"] = df.apply(check_caption_uniqueness, axis=1)

df.to_csv(output_path, index=False)

print(f"Saved to {output_path}")
print(df.head())

In [ ]:
import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer

In [ ]:
csv_path = "veri_image_captions_with_switch.csv"
output_path = "veri_image_captions.csv"

df = pd.read_csv(csv_path)
df["caption"] = df["caption"].fillna("").astype(str)

# Recreate helper columns if you dropped them earlier
df["condition"] = df["image_id"].str.extract(r"_(safe|danger)$")
df["base_id"] = df["image_id"].str.replace(r"_(safe|danger)$", "", regex=True)

def tokenize(text):
    return set(re.findall(r"\b\w+\b", text.lower()))

def jaccard_similarity(a, b):
    a_tokens = tokenize(a)
    b_tokens = tokenize(b)

    if not a_tokens or not b_tokens:
        return 0.0

    return len(a_tokens & b_tokens) / len(a_tokens | b_tokens)

def get_best_match(sim_scores, row_idx):
    current_base = df.at[row_idx, "base_id"]

    # Exclude itself and its safe/danger pair
    invalid = (df["base_id"] == current_base)

    scores = sim_scores.copy()
    scores[invalid.values] = -1

    best_idx = int(np.argmax(scores))
    return best_idx, scores[best_idx]

def get_match(sim_scores, row_idx, mode="max"):
    current_base = df.at[row_idx, "base_id"]

    # Exclude itself and its safe/danger pair
    invalid = (df["base_id"] == current_base)

    scores = sim_scores.copy()

    if mode == "max":
        scores[invalid.values] = -1
        match_idx = int(np.argmax(scores))
    elif mode == "min":
        scores[invalid.values] = np.inf
        match_idx = int(np.argmin(scores))
    else:
        raise ValueError("mode must be 'max' or 'min'")

    return match_idx, scores[match_idx]

# jaccard : most similiar word wise caption

df["most_similar_jaccard"] = ""
df["most_similar_jaccard_score"] = 0.0
df["least_similar_jaccard"] = ""
df["least_similar_jaccard_score"] = 0.0

for i in df.index:
    scores = np.array([
        jaccard_similarity(df.at[i, "caption"], other_caption)
        for other_caption in df["caption"]
    ])

    best_idx, best_score = get_match(scores, i, mode="max")
    worst_idx, worst_score = get_match(scores, i, mode="min")

    df.at[i, "most_similar_jaccard"] = df.at[best_idx, "caption"]
    df.at[i, "most_similar_jaccard_score"] = best_score
    df.at[i, "least_similar_jaccard"] = df.at[worst_idx, "caption"]
    df.at[i, "least_similar_jaccard_score"] = worst_score

# tf-idf : most similar based on tf-idf cosine similarity

vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform(df["caption"])

tfidf_sim = cosine_similarity(tfidf_matrix)

df["most_similar_tfidf"] = ""
df["most_similar_tfidf_score"] = 0.0
df["least_similar_tfidf"] = ""
df["least_similar_tfidf_score"] = 0.0

for i in df.index:
    best_idx, best_score = get_match(tfidf_sim[i], i, mode="max")
    worst_idx, worst_score = get_match(tfidf_sim[i], i, mode="min")

    df.at[i, "most_similar_tfidf"] = df.at[best_idx, "caption"]
    df.at[i, "most_similar_tfidf_score"] = best_score
    df.at[i, "least_similar_tfidf"] = df.at[worst_idx, "caption"]
    df.at[i, "least_similar_tfidf_score"] = worst_score

# sentence-BERT : most similar based on sentence embedding cosine similarity

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

embeddings = model.encode(
    df["caption"].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True
)

sbert_sim = cosine_similarity(embeddings)

df["most_similar_sbert"] = ""
df["most_similar_sbert_score"] = 0.0
df["least_similar_sbert"] = ""
df["least_similar_sbert_score"] = 0.0

for i in df.index:
    best_idx, best_score = get_match(sbert_sim[i], i, mode="max")
    worst_idx, worst_score = get_match(sbert_sim[i], i, mode="min")

    df.at[i, "most_similar_sbert"] = df.at[best_idx, "caption"]
    df.at[i, "most_similar_sbert_score"] = best_score
    df.at[i, "least_similar_sbert"] = df.at[worst_idx, "caption"]
    df.at[i, "least_similar_sbert_score"] = worst_score


# include safe/danger pairs

def get_most_similar_include_pair(sim_scores, row_idx):
    scores = sim_scores.copy()

    # Exclude only the exact same row/image
    scores[row_idx] = -1

    match_idx = int(np.argmax(scores))
    return match_idx, scores[match_idx]

# Jaccard: most similar, allowing safe/danger pair
df["most_similar_jaccard_include_pair"] = ""
df["most_similar_jaccard_include_pair_score"] = 0.0

for i in df.index:
    scores = np.array([
        jaccard_similarity(df.at[i, "caption"], other_caption)
        for other_caption in df["caption"]
    ])

    match_idx, score = get_most_similar_include_pair(scores, i)

    df.at[i, "most_similar_jaccard_include_pair"] = df.at[match_idx, "caption"]
    df.at[i, "most_similar_jaccard_include_pair_score"] = score


# TF-IDF: most similar, allowing safe/danger pair
df["most_similar_tfidf_include_pair"] = ""
df["most_similar_tfidf_include_pair_score"] = 0.0

for i in df.index:
    match_idx, score = get_most_similar_include_pair(tfidf_sim[i], i)

    df.at[i, "most_similar_tfidf_include_pair"] = df.at[match_idx, "caption"]
    df.at[i, "most_similar_tfidf_include_pair_score"] = score


# Sentence-BERT: most similar, allowing safe/danger pair
df["most_similar_sbert_include_pair"] = ""
df["most_similar_sbert_include_pair_score"] = 0.0

for i in df.index:
    match_idx, score = get_most_similar_include_pair(sbert_sim[i], i)

    df.at[i, "most_similar_sbert_include_pair"] = df.at[match_idx, "caption"]
    df.at[i, "most_similar_sbert_include_pair_score"] = score

# final stuff

df = df.drop(columns=["condition", "base_id"])

df.to_csv(output_path, index=False)

print(f"Saved to {output_path}")
print(df.head())